In [1]:
import json
import time
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.home() / "icidea_llm_ids"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

LABEL_MAP     = {0: "NORMAL", 1: "DOS", 2: "FUZZY", 3: "GEAR", 4: "RPM"}
LABEL_MAP_INV = {v: k for k, v in LABEL_MAP.items()}

SEED             = 42
SAMPLES_PER_CLASS = 500   # 300 test + 100 val + 100 reserve

t0 = time.time()
windows_df = pd.read_parquet(ARTIFACTS_DIR / "section7_windows_with_text.parquet")
print(f"✓ Loaded {len(windows_df):,} windows in {time.time()-t0:.2f}s")
print(f"  Class distribution:")
print(windows_df["label"].value_counts().sort_index().rename(LABEL_MAP).to_string())

✓ Loaded 195,069 windows in 0.92s
  Class distribution:
label
NORMAL    39998
DOS       39998
FUZZY     35077
GEAR      39998
RPM       39998


In [2]:
np.random.seed(SEED)

sampled_dfs = []
for lbl, name in LABEL_MAP.items():
    class_df = windows_df[windows_df["label"] == lbl]
    n = min(SAMPLES_PER_CLASS, len(class_df))
    sampled = class_df.sample(n=n, random_state=SEED)
    sampled_dfs.append(sampled)
    print(f"  {name:7s}  sampled {n} from {len(class_df):,}")

benchmark_df = pd.concat(sampled_dfs).reset_index(drop=True)
print(f"\nTotal benchmark windows: {len(benchmark_df):,}")

  NORMAL   sampled 500 from 39,998
  DOS      sampled 500 from 39,998
  FUZZY    sampled 500 from 35,077
  GEAR     sampled 500 from 39,998
  RPM      sampled 500 from 39,998

Total benchmark windows: 2,500


In [3]:
# Assign split within each class: 300 test / 100 val / 100 reserve
splits = []
for lbl in LABEL_MAP:
    class_idx = benchmark_df[benchmark_df["label"] == lbl].index.tolist()
    np.random.shuffle(class_idx)
    for i, idx in enumerate(class_idx):
        if i < 300:
            splits.append((idx, "test"))
        elif i < 400:
            splits.append((idx, "val"))
        else:
            splits.append((idx, "reserve"))

split_df = pd.DataFrame(splits, columns=["idx", "split"])
split_df = split_df.set_index("idx")
benchmark_df["split"] = split_df["split"]

print("Split distribution:")
print(benchmark_df.groupby(["split", "label"])
      .size()
      .unstack()
      .rename(columns=LABEL_MAP)
      .to_string())

Split distribution:
label    NORMAL  DOS  FUZZY  GEAR  RPM
split                                 
reserve     100  100    100   100  100
test        300  300    300   300  300
val         100  100    100   100  100


In [4]:
records = []
for _, row in benchmark_df.iterrows():
    frames = json.loads(row["frames_json"])
    records.append({
        "text":             row["text"],
        "label":            int(row["label"]),
        "label_name":       LABEL_MAP[int(row["label"])],
        "split":            row["split"],
        "window_start_ts":  row["window_start_ts"],
        "window_end_ts":    row["window_end_ts"],
        "frames_json":      row["frames_json"],
    })

final_df = pd.DataFrame(records)

print(f"Benchmark dataset: {len(final_df):,} windows")
print(f"\nSample record (test set, NORMAL):")
sample = final_df[(final_df["split"] == "test") & 
                  (final_df["label"] == 0)].iloc[0]
print(f"  label:      {sample['label']} ({sample['label_name']})")
print(f"  split:      {sample['split']}")
print(f"  text[:200]: {sample['text'][:200]}")

Benchmark dataset: 2,500 windows

Sample record (test set, NORMAL):
  label:      0 (NORMAL)
  split:      test
  text[:200]: CAN Bus Telemetry Sequence (14 frames):
[001] T=1479121529.933 ID=0545 DLC=8 DATA=D8 31 01 8E 00 00 00 00
[002] T=1479121529.935 ID=0153 DLC=8 DATA=00 00 00 FF 00 FF 00 00
[003] T=1479121529.936 ID=02


In [5]:
artifact_path = ARTIFACTS_DIR / "section8_benchmark.parquet"
final_df.to_parquet(artifact_path, index=False)
size_mb = artifact_path.stat().st_size / 1e6

print(f"✓ Saved benchmark to {artifact_path}")
print(f"  Size: {size_mb:.1f} MB")
print(f"  Total windows: {len(final_df):,}")
print(f"\nFinal split summary:")
print(final_df.groupby(["split", "label_name"])
      .size()
      .unstack()
      .to_string())

✓ Saved benchmark to /Users/deepakpatnaik/icidea_llm_ids/artifacts/section8_benchmark.parquet
  Size: 1.3 MB
  Total windows: 2,500

Final split summary:
label_name  DOS  FUZZY  GEAR  NORMAL  RPM
split                                    
reserve     100    100   100     100  100
test        300    300   300     300  300
val         100    100   100     100  100
